In [42]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path
import joblib

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.over_sampling import SMOTE

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)

In [43]:
PROJECT_ROOT = Path("..").resolve()

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "train_features.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test_features.csv"

MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(exist_ok=True)

In [44]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [45]:
X_train = train.drop(columns=["Machine_failure"])
y_train = train["Machine_failure"]

X_test = test.drop(columns=["Machine_failure"])
y_test = test["Machine_failure"]

In [46]:
numerical_features = [
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min"
]

In [47]:
scaler = StandardScaler()

X_train[numerical_features] = scaler.fit_transform(
    X_train[numerical_features]
)

X_test[numerical_features] = scaler.transform(
    X_test[numerical_features]
)

In [48]:
joblib.dump(
    scaler,
    MODEL_DIR / "scaler.pkl"
)

print("Scaler saved.")

Scaler saved.


In [49]:
joblib.dump(
    X_train.columns.tolist(),
    MODEL_DIR / "feature_columns.pkl"
)

print("Feature list saved.")

Feature list saved.


In [50]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [51]:
y_train_smote.value_counts()

Machine_failure
0    7729
1    7729
Name: count, dtype: int64

In [52]:
models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Random Forest":
        RandomForestClassifier(
            random_state=42
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        ),

    "LightGBM":
        LGBMClassifier(
            random_state=42
        ),

    "CatBoost":
        CatBoostClassifier(
            random_state=42,
            verbose=0
        )
}

In [53]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [54]:
results = []

trained_models = {}

In [55]:
for name, model in models.items():

    cv_score = cross_val_score(
        model,
        X_train_smote,
        y_train_smote,
        cv=cv,
        scoring="f1"
    ).mean()

    model.fit(
        X_train_smote,
        y_train_smote
    )

    trained_models[name] = model

    pred = model.predict(X_test)

    prob = model.predict_proba(X_test)[:, 1]

    results.append({

        "Model": name,

        "CV_F1": cv_score,

        "Accuracy":
            accuracy_score(
                y_test,
                pred
            ),

        "Precision":
            precision_score(
                y_test,
                pred
            ),

        "Recall":
            recall_score(
                y_test,
                pred
            ),

        "F1":
            f1_score(
                y_test,
                pred
            ),

        "ROC_AUC":
            roc_auc_score(
                y_test,
                prob
            ),

        "MCC":
            matthews_corrcoef(
                y_test,
                pred
            )

    })

[LightGBM] [Info] Number of positive: 6183, number of negative: 6183
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000713 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3839
[LightGBM] [Info] Number of data points in the train set: 12366, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 6183, number of negative: 6183
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000720 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3839
[LightGBM] [Info] Number of data points in the train set: 12366, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.00000

In [56]:
results = pd.DataFrame(results)

results.sort_values(
    by="F1",
    ascending=False
)

,Model,CV_F1,Accuracy,Precision,Recall,F1,ROC_AUC,MCC
2,Extra Trees,0.995537,0.9960,0.916667,0.970588,0.942857,0.994961,0.941199
4,LightGBM,0.995411,0.9940,0.868421,0.970588,0.916667,0.991749,0.915085
5,CatBoost,0.995083,0.9940,0.878378,0.955882,0.915493,0.994991,0.913265
1,Random Forest,0.992629,0.9925,0.835443,0.970588,0.897959,0.996198,0.896798
3,XGBoost,0.995215,0.9910,0.820513,0.941176,0.876712,0.993142,0.874276
0,Logistic Regression,0.953798,0.9670,0.507937,0.941176,0.659794,0.985865,0.678098


In [57]:
results

,Model,CV_F1,Accuracy,Precision,Recall,F1,ROC_AUC,MCC
0,Logistic Regression,0.953798,0.9670,0.507937,0.941176,0.659794,0.985865,0.678098
1,Random Forest,0.992629,0.9925,0.835443,0.970588,0.897959,0.996198,0.896798
2,Extra Trees,0.995537,0.9960,0.916667,0.970588,0.942857,0.994961,0.941199
3,XGBoost,0.995215,0.9910,0.820513,0.941176,0.876712,0.993142,0.874276
4,LightGBM,0.995411,0.9940,0.868421,0.970588,0.916667,0.991749,0.915085
5,CatBoost,0.995083,0.9940,0.878378,0.955882,0.915493,0.994991,0.913265


In [58]:
best_model_name = results.iloc[0]["Model"]

best_model = trained_models[
    best_model_name
]

print(best_model_name)

Logistic Regression


In [59]:
joblib.dump(
    best_model,
    MODEL_DIR / "best_model.pkl"
)

print("Best model saved.")

Best model saved.


In [60]:
results.to_csv(
    PROJECT_ROOT /
    "reports" /
    "model_results.csv",
    index=False
)

In [61]:
results

,Model,CV_F1,Accuracy,Precision,Recall,F1,ROC_AUC,MCC
0,Logistic Regression,0.953798,0.9670,0.507937,0.941176,0.659794,0.985865,0.678098
1,Random Forest,0.992629,0.9925,0.835443,0.970588,0.897959,0.996198,0.896798
2,Extra Trees,0.995537,0.9960,0.916667,0.970588,0.942857,0.994961,0.941199
3,XGBoost,0.995215,0.9910,0.820513,0.941176,0.876712,0.993142,0.874276
4,LightGBM,0.995411,0.9940,0.868421,0.970588,0.916667,0.991749,0.915085
5,CatBoost,0.995083,0.9940,0.878378,0.955882,0.915493,0.994991,0.913265


In [62]:
import joblib
from pathlib import Path


MODEL_DIR = Path("../models")

MODEL_DIR.mkdir(
    exist_ok=True
)


# Save all trained models

for name, model in trained_models.items():

    filename = (
        name
        .replace(" ", "_")
        .replace("-", "")
        + ".pkl"
    )

    joblib.dump(
        model,
        MODEL_DIR / filename
    )

    print(f"{name} saved")


# Save best model (Extra Trees)

best_model = trained_models["Extra Trees"]

joblib.dump(
    best_model,
    MODEL_DIR / "best_model.pkl"
)


print("Best model saved")


# Save feature names

joblib.dump(
    X_train.columns.tolist(),
    MODEL_DIR / "feature_names.pkl"
)


print("Feature names saved")


# Save scaler

joblib.dump(
    scaler,
    MODEL_DIR / "scaler.pkl"
)


print("Scaler saved")

Logistic Regression saved
Random Forest saved
Extra Trees saved
XGBoost saved
LightGBM saved
CatBoost saved
Best model saved
Feature names saved
Scaler saved
